# Load the required libraries

In [ ]:
library(CellChat)
library(Seurat)
library(dplyr)
library(patchwork)
library(Matrix)
options(stringsAsFactors = FALSE)
options(future.globals.maxSize = 8 * 1024^3)  # 8GB 

Warning message:
“package ‘CellChat’ was built under R version 4.4.3”


Loading required package: dplyr


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: igraph


Attaching package: ‘igraph’


The following objects are masked from ‘package:dplyr’:

    as_data_frame, groups, union


The following objects are masked from ‘package:stats’:

    decompose, spectrum


The following object is masked from ‘package:base’:

    union


Loading required package: ggplot2

Loading required package: SeuratObject

Warning message:
“package ‘SeuratObject’ was built under R version 4.4.3”
Loading required package: sp

‘SeuratObject’ was built with package ‘Matrix’ 1.7.2 but the current
version is 1.7.3; it is recomended that you reinstall ‘SeuratObject’ as
the ABI for ‘Matrix’ may have changed


Attaching package: ‘SeuratObject’


The following object is masked from ‘package:BiocGenerics’:

    in

# Part I: Data input & processing and initialization of CellChat object

In [7]:
rna <- readRDS("/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/multimapping/GW7/spinalcord-rna-GW7.final.rds")
rna

An object of class Seurat 
27130 features across 11955 samples within 1 assay 
Active assay: RNA (27130 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 3 dimensional reductions calculated: pca, umap, tsne

In [8]:
# -----------(B) Starting from a Seurat object 
data.input <- rna[["RNA"]]@data # normalized data matrix
# check how many !0 genes
nonzero_genes <- Matrix::rowSums(data.input) > 0
filtered_mat <- data.input[nonzero_genes, ]
#dim(filtered_mat) 
# For Seurat version >= “5.0.0”, get the normalized data via `seurat_object[["RNA"]]$data`

cellType <- c(
    "APC_p0/1_progenitor", #APC    
    "dp3-6", #Trip
    "multipotential_NPC", "NPC_proliferative", "p0/1", "to_pMN_proliferative?", "dp1/2", # Neural_progenitor
    "MN-1", "MN-2", "MN-3", #Motor_neuron
    "di1-1", "di1-2", "di1-3", "di2_or_di3/5-1", "di3/5-2", "di3/5-3?", "di3/5-4?", "di4/6?", "v2a", "v3", "unknow2", "unknow3", # Excitatory
    "di4/6-1", "di4/6-2", "di4/6-3", "v0/1", "v2b", #Inhibitory
    "Floor_plate", "Roof_plate", #Ependymal
    "Microglial", #Microglia
    "unknow1" #Unknown
)
# define the meta data: 
# a column named `samples` should be provided for spatial transcriptomics analysis, which is useful for analyzing cell-cell communication by aggregating multiple samples/replicates. Of note, for comparison analysis across different conditions, users still need to create a CellChat object seperately for each condition.  
Idents(rna) <- "cell_name"
labels <- Idents(rna)
meta = data.frame(labels = Seurat::Idents(rna), samples = "gw7", row.names = names(Seurat::Idents(rna))) # manually create a dataframe consisting of the cell labels
meta$labels <- factor(meta$labels, levels = cellType)
meta$samples <- factor(meta$samples)

# create a cellchat object
cellchat <- createCellChat(object = filtered_mat, meta = meta, group.by = "labels")
#> [1] "Create a CellChat object from a data matrix"
#An object of class CellChat created from a single dataset 
# 20518 genes.
# 3682 cells. 
#CellChat analysis of single cell RNA-seq data! 

[1] "Create a CellChat object from a data matrix"
Set cell identities for the new CellChat object 
The cell groups used for CellChat analysis are  APC_p0/1_progenitor, dp3-6, multipotential_NPC, NPC_proliferative, p0/1, to_pMN_proliferative?, dp1/2, MN-1, MN-2, MN-3, di1-1, di1-2, di1-3, di2_or_di3/5-1, di3/5-2, di3/5-3?, di3/5-4?, di4/6?, v2a, v3, unknow2, unknow3, di4/6-1, di4/6-2, di4/6-3, v0/1, v2b, Floor_plate, Roof_plate, Microglial, unknow1 


In [9]:
groupSize <- as.numeric(table(cellchat@idents)) 
# Set the ligand-receptor interaction database
CellChatDB <- CellChatDB.human # use CellChatDB.mouse if running on mouse data
#showDatabaseCategory(CellChatDB)
# use a subset of CellChatDB for cell-cell communication analysis
#CellChatDB.use <- subsetDB(CellChatDB, search = "Secreted Signaling", key = "annotation") # use Secreted Signaling
# Only uses the Secreted Signaling from CellChatDB v1
#  CellChatDB.use <- subsetDB(CellChatDB, search = list(c("Secreted Signaling"), c("CellChatDB v1")), key = c("annotation", "version"))
# use all CellChatDB except for "Non-protein Signaling" for cell-cell communication analysis
# CellChatDB.use <- subsetDB(CellChatDB)
# use all CellChatDB for cell-cell communication analysis
CellChatDB.use <- CellChatDB # simply use the default CellChatDB. We do not suggest to use it in this way because CellChatDB v2 includes "Non-protein Signaling" (i.e., metabolic and synaptic signaling). 
# set the used database in the object
cellchat@DB <- CellChatDB

# subset the expression data of signaling genes for saving computation cost
cellchat <- subsetData(cellchat) # This step is necessary even if using the whole database
future::plan("multisession", workers = 4) # do parallel
cellchat <- identifyOverExpressedGenes(cellchat)
cellchat <- identifyOverExpressedInteractions(cellchat)

The number of highly variable ligand-receptor pairs used for signaling inference is 2591 


# Part II: Inference of cell-cell communication network

In [10]:
#Compute the communication probability and infer cellular communication network
cellchat <- computeCommunProb(cellchat, type = "triMean") %>% # type = "truncatedMean", trim = 0.1
  #filterCommunication(min.cells = 10) %>% 
  computeCommunProbPathway() %>% # Infer the cell-cell communication at a signaling pathway level
  aggregateNet() # Calculate the aggregated cell-cell communication network
cellchat <- netAnalysis_computeCentrality(cellchat, slot.name = "netP")

triMean is used for calculating the average gene expression per cell group. 
[1] ">>> Run CellChat on sc/snRNA-seq data <<< [2025-05-29 19:36:59.850663]"
[1] ">>> CellChat inference is done. Parameter values are stored in `object@options$parameter` <<< [2025-05-29 20:39:32.690784]"


In [12]:
cellchat@netP$pathways

[1] "NRXN"        "ADGRL"       "SLIT"        "NRG"         "NCAM"       
 [6] "PTPR"        "Glutamate"   "CADM"        "EPHA"        "NEGR"       
[11] "Netrin"      "CDH"         "CNTN"        "PTPRM"       "NGL"        
[16] "PTN"         "SEMA3"       "UNC5"        "JAM"         "SEMA6"      
[21] "EPHB"        "SLITRK"      "APP"         "BMP"         "CD46"       
[26] "FLRT"        "CD99"        "COLLAGEN"    "NT"          "MK"         
[31] "NOTCH"       "LAMININ"     "Cholesterol" "CypA"        "SPP1"       
[36] "ncWNT"       "RELN"        "FGF"         "WNT"         "TENASCIN"   
[41] "GABA-B"      "GABA-A"      "CD45"        "MPZ"         "TAC"        
[46] "GAP"

## summary plot(all interactions/seperate interactions)

In [ ]:
# all interactions merged!!!
groupSize <- as.numeric(table(cellchat@idents))
par(mfrow = c(1,2), xpd=TRUE)
netVisual_circle(cellchat@net$count, vertex.weight = groupSize, weight.scale = T, label.edge= F, title.name = "Number of interactions")
netVisual_circle(cellchat@net$weight, vertex.weight = groupSize, weight.scale = T, label.edge= F, title.name = "Interaction weights/strength")

In [ ]:
# seperate interactions by each cell group!!!
mat <- cellchat@net$weight
par(mfrow = c(3,4), xpd=TRUE)
for (i in 1:nrow(mat)) {
  mat2 <- matrix(0, nrow = nrow(mat), ncol = ncol(mat), dimnames = dimnames(mat))
  mat2[i, ] <- mat[i, ]
  netVisual_circle(mat2, vertex.weight = groupSize, weight.scale = T, edge.weight.max = max(mat), title.name = rownames(mat)[i])
}

## plot

In [ ]:
# Zoom in each signaling pathway
# Circle plot
pathways.show <- c("GABA-A")
netVisual_aggregate(cellchat, signaling = pathways.show, layout = "circle",  top = 0.005, remove.isolate = F)

# Chord diagram
par(mfrow=c(1,1))
netVisual_aggregate(cellchat, signaling = pathways.show, layout = "chord")

# Heatmap
par(mfrow=c(1,1))
netVisual_heatmap(cellchat, signaling = pathways.show, color.heatmap = "Reds")

# ligand-receptor pair
netAnalysis_contribution(cellchat, signaling = pathways.show)

# Bubble plot
par(mfrow=c(1,1))
netVisual_bubble(cellchat, sources.use = 9, targets.use = c(1:31), signaling = pathways.show, remove.isolate = FALSE)

# signaling gene expression
plotGeneExpression(cellchat, signaling = pathways.show, enriched.only = TRUE, type = "violin")

# network centrality scores
netAnalysis_signalingRole_network(cellchat, signaling = pathways.show, width = 12, height = 2.5, font.size = 10)


In [32]:
source('/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/CCC/report_plot.R')

In [33]:
# Define parameter list
my_params <- list(
  pathways.show = c("GABA-B"),  # Multiple pathways can be specified
  sources.use = 27,             # Optional parameter
  targets.use = c(1:31),       # Optional parameter
  top = 0.005                  # Optional parameter
)

# Generate report
generate_cellchat_report(
  cellchat = cellchat,  # Your CellChat object
  params = my_params,
  output_dir = "./plot"
)

Comparing communications on a single object 


Scale for y is already present.
Adding another scale for y, which will replace the existing scale.
Scale for y is already present.
Adding another scale for y, which will replace the existing scale.
Scale for y is already present.
Adding another scale for y, which will replace the existing scale.
Scale for y is already present.
Adding another scale for y, which will replace the existing scale.
Scale for y is already present.
Adding another scale for y, which will replace the existing scale.
Report saved to: /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/CCC/GW7/plot/CellChat_GABA_B_Report.pdf

Do heatmap based on a single object 


Heatmap also saved separately to: /cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ST/CCC/GW7/plot/Heatmap_GABA_B.pdf



## Check evidence

In [38]:
df.net <- subsetCommunication(cellchat)

In [65]:
path.net <- df.net[df.net$pathway_name == 'GABA-A',]
path.net

,source,target,ligand,receptor,prob,pval,interaction_name,interaction_name_2,pathway_name,annotation,evidence
,<fct>,<fct>,<chr>,<chr>,<dbl>,<dbl>,<fct>,<chr>,<chr>,<chr>,<chr>
4858,v2b,MN-1,GABA-GAD1_SLC6A1,GABR_A3B3G2,0.005215637,0,GABA-A-GAD1_SLC6A1_GABR_A3B3G2,GABA-A-(GAD1+SLC6A1) - GABR_A3B3G2,GABA-A,Non-protein Signaling,PMID: 34417930;PMID: 23038269
4859,v2b,MN-2,GABA-GAD1_SLC6A1,GABR_A3B3G2,0.005597978,0,GABA-A-GAD1_SLC6A1_GABR_A3B3G2,GABA-A-(GAD1+SLC6A1) - GABR_A3B3G2,GABA-A,Non-protein Signaling,PMID: 34417930;PMID: 23038269
4860,v2b,MN-1,GABA-GAD2_SLC6A1,GABR_A3B3G2,0.005422667,0,GABA-A-GAD2_SLC6A1_GABR_A3B3G2,GABA-A-(GAD2+SLC6A1) - GABR_A3B3G2,GABA-A,Non-protein Signaling,PMID: 34417930;PMID: 23038269
4861,v2b,MN-2,GABA-GAD2_SLC6A1,GABR_A3B3G2,0.005820096,0,GABA-A-GAD2_SLC6A1_GABR_A3B3G2,GABA-A-(GAD2+SLC6A1) - GABR_A3B3G2,GABA-A,Non-protein Signaling,PMID: 34417930;PMID: 23038269


In [ ]:
table(df.net$pathway_name)


      ADGRL         APP         BMP        CADM        CD45        CD46 
       3848         217         174         605           1         155 
       CD99         CDH Cholesterol        CNTN    COLLAGEN        CypA 
         90         361          31         650          75          60 
       EPHA        EPHB         FGF        FLRT      GABA-A      GABA-B 
       1532         603          30         144           4          12 
        GAP   Glutamate         JAM     LAMININ          MK         MPZ 
          1        6959         583          66         138           9 
       NCAM       ncWNT        NEGR      Netrin         NGL       NOTCH 
       1079          33         425         434         391          96 
        NRG        NRXN          NT         PTN        PTPR       PTPRM 
        874        3855          92         559        1953         364 
       RELN       SEMA3       SEMA6        SLIT      SLITRK        SPP1 
         17         707         645        1315   

## Search specific cell type

In [40]:
head(df.net)

,source,target,ligand,receptor,prob,pval,interaction_name,interaction_name_2,pathway_name,annotation,evidence
,<fct>,<fct>,<chr>,<chr>,<dbl>,<dbl>,<fct>,<chr>,<chr>,<chr>,<chr>
1,Roof_plate,APC_p0/1_progenitor,GDF7,BMPR1A_ACVR2A,0.01298189,0,GDF7_BMPR1A_ACVR2A,GDF7 - (BMPR1A+ACVR2A),BMP,Secreted Signaling,KEGG: hsa04350; PMID:26893264
2,Roof_plate,dp3-6,GDF7,BMPR1A_ACVR2A,0.02351487,0,GDF7_BMPR1A_ACVR2A,GDF7 - (BMPR1A+ACVR2A),BMP,Secreted Signaling,KEGG: hsa04350; PMID:26893264
3,Roof_plate,multipotential_NPC,GDF7,BMPR1A_ACVR2A,0.01278808,0,GDF7_BMPR1A_ACVR2A,GDF7 - (BMPR1A+ACVR2A),BMP,Secreted Signaling,KEGG: hsa04350; PMID:26893264
4,Roof_plate,NPC_proliferative,GDF7,BMPR1A_ACVR2A,0.01841957,0,GDF7_BMPR1A_ACVR2A,GDF7 - (BMPR1A+ACVR2A),BMP,Secreted Signaling,KEGG: hsa04350; PMID:26893264
5,Roof_plate,p0/1,GDF7,BMPR1A_ACVR2A,0.01064243,0,GDF7_BMPR1A_ACVR2A,GDF7 - (BMPR1A+ACVR2A),BMP,Secreted Signaling,KEGG: hsa04350; PMID:26893264
6,Roof_plate,to_pMN_proliferative?,GDF7,BMPR1A_ACVR2A,0.01535148,0,GDF7_BMPR1A_ACVR2A,GDF7 - (BMPR1A+ACVR2A),BMP,Secreted Signaling,KEGG: hsa04350; PMID:26893264


In [68]:
# Use dplyr to compute the 20th percentile for each pathway (top 80% cutoff)
df.net <- df.net %>%
  group_by(pathway_name) %>%
  mutate(cutoff_0.8 = quantile(prob, probs = 0.2, na.rm = TRUE)) %>%
  add_count(name = "frequence") %>%  
  ungroup()
head(df.net)

In [70]:
# Filter specific source and target
vtomn <- df.net[df.net$source %in% c("v0/1", "v2a", "v2b", "v3") & df.net$target %in% c("MN-1", "MN-2", "MN-3") & df.net$prob > df.net$cutoff_0.8, ]
#vtomn <- vtomn[order(vtomn$prob, decreasing = TRUE), ]
head(vtomn)

source,target,ligand,receptor,prob,pval,interaction_name,interaction_name_2,pathway_name,annotation,evidence,cutoff_0.1,frequence
<fct>,<fct>,<chr>,<chr>,<dbl>,<dbl>,<fct>,<chr>,<chr>,<chr>,<chr>,<dbl>,<int>
v2a,MN-3,PTN,ALK,0.011458330,0,PTN_ALK,PTN - ALK,PTN,Secreted Signaling,PMID: 28356350; PMID: 25620911,0.004527746,559
v0/1,MN-3,PTN,ALK,0.014136156,0,PTN_ALK,PTN - ALK,PTN,Secreted Signaling,PMID: 28356350; PMID: 25620911,0.004527746,559
v2b,MN-3,PTN,ALK,0.015271428,0,PTN_ALK,PTN - ALK,PTN,Secreted Signaling,PMID: 28356350; PMID: 25620911,0.004527746,559
v2a,MN-1,SEMA3A,NRP1_PLXNA2,0.010242466,0,SEMA3A_NRP1_PLXNA2,SEMA3A - (NRP1+PLXNA2),SEMA3,Secreted Signaling,PMID: 27533782,0.005905415,707
v0/1,MN-1,SEMA3A,NRP1_PLXNA2,0.011131589,0,SEMA3A_NRP1_PLXNA2,SEMA3A - (NRP1+PLXNA2),SEMA3,Secreted Signaling,PMID: 27533782,0.005905415,707
v2b,MN-1,SEMA3A,NRP1_PLXNA2,0.007550883,0,SEMA3A_NRP1_PLXNA2,SEMA3A - (NRP1+PLXNA2),SEMA3,Secreted Signaling,PMID: 27533782,0.005905415,707


In [75]:
pathway_stats <- vtomn %>%
  group_by(pathway_name) %>%
  summarise(
    count = n(), 
    frequence = first(frequence),
    ratio = count / frequence
  ) %>%
  filter(ratio >= 0.2) 
pathway_stats

In [78]:
vtomn <- vtomn %>%
  filter(pathway_name %in% pathway_stats$pathway_name)
vtomn

source,target,ligand,receptor,prob,pval,interaction_name,interaction_name_2,pathway_name,annotation,evidence,cutoff_0.1,frequence
<fct>,<fct>,<chr>,<chr>,<dbl>,<dbl>,<fct>,<chr>,<chr>,<chr>,<chr>,<dbl>,<int>
v2b,MN-2,GABA-GAD1_SLC6A1,GABR_A3B3G2,0.005597978,0,GABA-A-GAD1_SLC6A1_GABR_A3B3G2,GABA-A-(GAD1+SLC6A1) - GABR_A3B3G2,GABA-A,Non-protein Signaling,PMID: 34417930;PMID: 23038269,0.005339855,4
v2b,MN-1,GABA-GAD2_SLC6A1,GABR_A3B3G2,0.005422667,0,GABA-A-GAD2_SLC6A1_GABR_A3B3G2,GABA-A-(GAD2+SLC6A1) - GABR_A3B3G2,GABA-A,Non-protein Signaling,PMID: 34417930;PMID: 23038269,0.005339855,4
v2b,MN-2,GABA-GAD2_SLC6A1,GABR_A3B3G2,0.005820096,0,GABA-A-GAD2_SLC6A1_GABR_A3B3G2,GABA-A-(GAD2+SLC6A1) - GABR_A3B3G2,GABA-A,Non-protein Signaling,PMID: 34417930;PMID: 23038269,0.005339855,4
v2b,MN-1,GABA-GAD1_SLC6A1,GABBR2,0.011029426,0,GABA-B-GAD1_SLC6A1_GABBR2,GABA-B-(GAD1+SLC6A1) - GABBR2,GABA-B,Non-protein Signaling,PMID: 32871173;PMID: 30541966,0.003672702,12
v2b,MN-2,GABA-GAD1_SLC6A1,GABBR2,0.003808377,0,GABA-B-GAD1_SLC6A1_GABBR2,GABA-B-(GAD1+SLC6A1) - GABBR2,GABA-B,Non-protein Signaling,PMID: 32871173;PMID: 30541966,0.003672702,12
v2b,MN-1,GABA-GAD2_SLC6A1,GABBR2,0.011464569,0,GABA-B-GAD2_SLC6A1_GABBR2,GABA-B-(GAD2+SLC6A1) - GABBR2,GABA-B,Non-protein Signaling,PMID: 32871173;PMID: 30541966,0.003672702,12
v2b,MN-2,GABA-GAD2_SLC6A1,GABBR2,0.003959769,0,GABA-B-GAD2_SLC6A1_GABBR2,GABA-B-(GAD2+SLC6A1) - GABBR2,GABA-B,Non-protein Signaling,PMID: 32871173;PMID: 30541966,0.003672702,12
v2b,MN-3,GABA-GAD2_SLC6A1,GABBR2,0.003792644,0,GABA-B-GAD2_SLC6A1_GABBR2,GABA-B-(GAD2+SLC6A1) - GABBR2,GABA-B,Non-protein Signaling,PMID: 32871173;PMID: 30541966,0.003672702,12


# Save and Load results

In [11]:
saveRDS(cellchat, file = "cellchat_snrna_human_spine_gw7.rds")

In [2]:
cellchat <- readRDS("cellchat_snrna_human_spine_gw7.rds")